# 01 — Build dataset (canonical data-construction path)

This notebook is the **single source of truth** for `df_raw` and the pilot
`panel`. Every other notebook should replace its inline data cells with either

```python
%run notebooks/01_build_dataset.ipynb
```

or an import of the same `src` functions, and obtain an **identical** `df_raw`
and `panel`. All loading / feature / audit logic now lives in the `src` package;
this notebook only wires it together and prints the usual reports.


## Step 0 — Make `src` importable and pin paths

Search upward from the working directory for the folder containing the `src`
package, so the notebook works whether it is opened directly (cwd = `notebooks/`)
or `%run` from the parent `Script/` directory. `DATA_PATH` honours the
`THESIS_DATA_PATH` env var, else defaults to `<root>/Data`.


In [ ]:
import os, sys

# Walk up from cwd until we find the dir that holds the src/ package.
_ROOT = os.path.abspath(os.getcwd())
while _ROOT != os.path.dirname(_ROOT):
    if os.path.isdir(os.path.join(_ROOT, 'src', 'data')):
        break
    _ROOT = os.path.dirname(_ROOT)
if _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)

DATA_PATH = os.environ.get('THESIS_DATA_PATH', os.path.join(_ROOT, 'Data'))
# Cache-first NOR panel: read the committed cache, no network by default.
NOR_CACHE = os.path.join(_ROOT, 'htboost_results_v5_nor', 'norway_raw_features.csv')

# Run parameters (mirror the v5 notebook's CONFIG for the data-build portion).
UNIVERSE_MIN_OBS = 500
SMOKE_SEC        = 'NOR_10Y'
H                = 5

print(f'src root:   {_ROOT}')
print(f'DATA_PATH:  {DATA_PATH}')
print(f'NOR_CACHE:  {NOR_CACHE}')


## Step 1 — Imports from `src`

Loaders, feature builders, universe/panel helpers and the leakage audit — every
function that used to be defined inline in the model notebooks.


In [ ]:
from src.data.bloomberg import load_data
from src.data.norway import load_norway_raw, print_connectivity_report
from src.features.macro import add_macro_features
from src.features.norway import add_norway_features
from src.features.security import features_for_security
from src.features.panel import build_panel, swap_columns, build_universe
from src.audit.leakage import leakage_audit


## Step 2 — Load Bloomberg swap/rate data

Merge the per-country swap CSVs plus the interest-rate and macro/market files.
`Duration.xlsx` is never loaded by `load_data` — we assert no duration columns
leaked in — then report the shape, date range and swap-column count.


In [ ]:
df_raw = load_data(DATA_PATH)

SWAP_COLS = swap_columns(df_raw)

# Duration.xlsx is NOT loaded by load_data() — guard anyway
dur_cols = [c for c in df_raw.columns if 'uration' in c.lower() or 'DV01' in c]
assert len(dur_cols) == 0, f'Duration columns found — exclude: {dur_cols}'

print(f'df_raw shape:    {df_raw.shape}')
print(f'Date range:      {df_raw.index.min().date()} → {df_raw.index.max().date()}')
print(f'Swap columns:    {len(SWAP_COLS)}')
print(f'Non-swap columns:{len(df_raw.columns) - len(SWAP_COLS)}')
print(f'Duration guard:  OK (0 duration columns in df_raw)')


## Step 3 — Load Norway raw series (cache-first)

`load_norway_raw(..., live=False)` reads the committed cache so the panel is
reproducible across machines. Pass `live=True` to refresh from the public
Norges Bank / SSB / ECB / Riksbank endpoints (and rewrite the cache). The
connectivity report uses the preserved 7-tuple format.


In [ ]:
nor_raw, nor_report = load_norway_raw(df_raw.index.min(), df_raw.index.max(),
                                      NOR_CACHE, live=False)
print_connectivity_report(nor_report)
print(f'\nnor_raw shape: {nor_raw.shape}')


## Step 4 — Join daily NOR series and derive `nor_*` features

Daily raw series (FX, policy rate, NOWA, govt yields, ECB/Riksbank) are joined
into `df_raw`; the monthly SSB series stay in `nor_raw` (native month index).
`add_norway_features` then derives the leakage-safe `nor_*` columns (FX shift(1);
policy/NOWA/govt contemporaneous; monthly ffill+shift(1)). The join is
idempotent — only columns not already present are merged.


In [ ]:
_daily_raw = [c for c in nor_raw.columns if not c.startswith('ssb_')]
_join_cols = [c for c in _daily_raw if c not in df_raw.columns]
if _join_cols:
    df_raw = df_raw.join(nor_raw[_join_cols], how='left')

df_raw = add_norway_features(df_raw, nor_raw)
NOR_FEATURE_COLS = sorted(c for c in df_raw.columns if c.startswith('nor_'))
print(f'{len(NOR_FEATURE_COLS)} nor_* feature columns derived (gated to country==\'NOR\'):')
for c in NOR_FEATURE_COLS:
    print(f'  {c:30s} {df_raw[c].notna().sum():6d} obs')


## Step 5 — Add macro / global features

Derive the macro, volatility, equity, commodity, credit, inflation and
country-IBOR columns. Timing is baked in here (US-close series shift(1), IBOR
contemporaneous, monthly ffill+shift(1)). `nor_*` columns from Step 4 are
preserved.


In [ ]:
_cols_before = set(df_raw.columns)
df_raw = add_macro_features(df_raw)
print(f'df_raw augmented  →  shape: {df_raw.shape}')

new_cols = sorted(c for c in df_raw.columns if c not in _cols_before)
print(f'\nDerived macro/global columns added ({len(new_cols)}):')
print(f'  {"Column":<35s}  {"n_obs":>6s}')
print('  ' + '-' * 44)
for c in new_cols:
    print(f'  {c:<35s}  {df_raw[c].notna().sum():6d}')

_n_nor = sum(1 for c in df_raw.columns if c.startswith('nor_'))
print(f'\nNorway nor_* feature columns present after augmentation: {_n_nor}')


## Step 6 — Build the NOR universe

Freeze the universe on full history (swaps with ≥ `UNIVERSE_MIN_OBS` obs), then
apply **PART A**: restrict to Norwegian swaps. `SMOKE_SEC` must survive.


In [ ]:
UNIVERSE = build_universe(df_raw, SWAP_COLS, UNIVERSE_MIN_OBS)
print(f'Universe (≥{UNIVERSE_MIN_OBS} obs): {len(UNIVERSE)} securities')

# PART A: restrict universe to Norwegian swaps
UNIVERSE = [s for s in UNIVERSE if s.rsplit('_', 1)[0] == 'NOR']
print(f'\nPART A — Norwegian-swap universe ({len(UNIVERSE)} securities):')
for s in UNIVERSE:
    print(f'  {s:<10s}  {df_raw[s].notna().sum():5d} obs')
assert SMOKE_SEC in UNIVERSE, f'{SMOKE_SEC} not in NOR universe'


## Step 7 — Build a pilot panel and run the leakage audit

Build the stacked panel for `SMOKE_SEC` (target `y = level.shift(-H) - level`),
then audit: recompute each sampled feature from past-only data and confirm it
matches the stored value within `1e-6`. The build fails loudly if any feature
leaks.


In [ ]:
panel = build_panel(df_raw, [SMOKE_SEC], H)
print(f'Pilot panel for audit: {panel.shape}\n')

_audit_ok = leakage_audit(panel, df_raw, n_samples=30)
assert _audit_ok, 'Leakage audit failed — fix before proceeding'
print('\n[OK] df_raw and panel are ready for downstream notebooks.')
